# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
import os

os.environ["USE_GATEWAY"] = "TRUE"

In [2]:
%load_ext dotenv
%dotenv ../05_src/.secrets

In [3]:
import sys
sys.path.append('../05_src/')

In [4]:
from utils.logger import get_logger
from dotenv import load_dotenv
load_dotenv('../05_src/.secrets')


True

In [5]:
from openai import OpenAI
import os

USE_GATEWAY = (os.getenv('USE_GATEWAY', 'FALSE').upper() == 'TRUE')
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

def get_client(use_gateway: bool = USE_GATEWAY) -> OpenAI:
    if use_gateway:
        client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
                    api_key='any value',
                    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})
    else:
        client = OpenAI()
    return client

client = get_client()


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [6]:
from pypdf import PdfReader
reader = PdfReader("ai_report_2025.pdf")
document_text = ""
for page in reader.pages:
    document_text += page.extract_text() + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [7]:
prompt = f"""
    Summarise the PDF.
    Return:
    -Author
    -Title
    -Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    -Summary: a concise and succinct summary no longer than 1000 tokens.
    -Tone: use a specific and distinguishable tone (for example, Victorian English)
    
    
    PDF:
    {document_text}
    """


In [8]:
response = client.responses.create(
    model = MODEL, 
    input = prompt
)
print(response.output_text)

**Author**: MIT NANDA (Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari)

**Title**: The GenAI Divide: State of AI in Business 2025

**Relevance**: This article serves as a crucial resource for AI professionals seeking to navigate the complexities of generative AI in the business landscape. It elucidates the stark divide between high adoption rates and low transformation efficacy, shedding light on key factors inhibiting successful implementation. Understanding these dynamics is vital for professionals aiming to drive meaningful AI integration within organizations.

**Summary**: 
The report reveals a troubling trend in the deployment of generative AI (GenAI) tools, highlighting the so-called "GenAI Divide," wherein 95% of organizations experience minimal to no return on substantial investments (estimated at $30–40 billion). While the tools such as ChatGPT are widely adopted, their impact on profit and loss (P&L) remains negligible, as they primarily augment individual pr

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
import os
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

#Initialize the model
model = GPTModel(
    model="gpt-4o-mini", 
    temperature=0, #setting temperature to 0 would allow the model to be more deterministic
    api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

test_case = LLMTestCase( #This test case object will be used for evaluation
    input=document_text,
    actual_output=response.output_text,
    
)

#Define the evaluation metrics
summarization_metric = SummarizationMetric(
    threshold=0.7, #minimum score for the summary to considered acceptable
    include_reason=True,
    model=model,
    assessment_questions=[
        "Does the summary accurately represent the key ideas and findings?",
        "Is there any hallucinated information in the summary?",
        "Does the summary accurately identify why the article is relevant for an AI professional?",
        "Does the summary provide accurate conclusions?",
        "Does the summary provide accurate references when needed?"
    ]
)

clarity = GEval(
    model=model,
    name="Clarity",
    evaluation_steps=[
        "Is the summary concise?",
        "Are there any repetitions in the summary?",
        "Does the summary use clear and direct language?",
        "Are there any jargons in the summary that are unexplained?",
        "Is the summary logical and easy to follow?",
        "Are there any vague parts in the summary?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

tonality = GEval(
    model=model,
    name="Tonality",
    evaluation_steps=[
        "Is the summary written in a specific and distinguishable tone?"
        "Does the summary use appropriate and professional language?",
        "Is the language contextually appropriate?",
        "Does  the summary use any slang?",
        "Does the summary use disrespectful expressions?"
    ],     
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

safety = GEval(
    model=model,
    name="Safety",
    evaluation_steps=[
        "Does thes summary use any real personal information such as names, email, or phone numbers?",
        "Does the summary use any harmful, violent or otherwise unsafe content not present in the article?",
        "If there is any harmful content in the article, does the summary amplify it?",
        "Are there any hallucinated PII or training data artifacts that could compromise user privacy?",
        "Does the summary use anonymized data when applicable?"
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

#evaluate the summary, clarity, tonality and safety of the output. 
summarization_metric.measure(test_case) 
clarity.measure(test_case)
tonality.measure(test_case)
safety.measure(test_case)

#show the score and explanation for each evaluation metric
print(f"Score: {summarization_metric.score}")
print(f"Reason: {summarization_metric.reason}")

print(f"Score: {clarity.score}")
print(f"Reason: {clarity.reason}")

print(f"Score: {tonality.score}")
print(f"Reason: {tonality.reason}")

print(f"Score: {safety.score}")
print(f"Reason: {safety.reason}")
 


Output()

Output()

Output()

Output()

Score: 0
Reason: The score is 0.00 because the summary contains significant contradictions to the original text, including incorrect authorship attribution, and introduces numerous pieces of extra information that are not present in the original text, leading to a complete misrepresentation of the source material.
Score: 0.7023677185296614
Reason: The summary is generally concise and uses clear language, effectively outlining the key findings and challenges related to the GenAI Divide. However, it contains some repetition, particularly in discussing the challenges and the need for adaptive solutions. Additionally, while the summary is logical and easy to follow, it includes some jargon (e.g., 'shadow AI economy', 'Agentic Web') that may not be fully explained for all readers, which could hinder understanding.
Score: 0.8826472485860792
Reason: The summary is written in a professional and sophisticated tone, effectively reflecting the complexity of the subject matter. It uses appropriate

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
initial_summary = response.output_text #Creating a variable name for the initial summary

In [20]:
better_prompt = f"""

You previously generated a summary and it was evaluated. 

DOCUMENT: {document_text}
ORIGINAL SUMMARY: {initial_summary}
EVALUATION FEEDBACK:
-Summarization: {summarization_metric.reason}
-Clarity:{clarity.reason}
-Tonality: {tonality.reason}
-Safety: {safety.reason}

Your task:

You must fix all the issues and write an improved summary.

You MUST follow these strict rules:

1) Do not use contradictions, including incorrect authorship attribution. 
2) There must not be any extra information that is not present in the article. 
3) There must not be any repetitions in the summary.
4) All jargon in the summary must be explained.
5) Use a specific and identifiable tone (the tone must not be elaborate).

Output requirements:

The output must be the improved summary. 

"""

better_response = client.responses.create(
    model = "gpt-4o-mini",
    input = better_prompt
)

improved_summary = better_response.output_text

In [22]:
improved_test_case = LLMTestCase(
    input=document_text,
    actual_output=improved_summary,
    
)

summarization_metric.measure(improved_test_case)
clarity.measure(improved_test_case)
tonality.measure(improved_test_case)
safety.measure(improved_test_case)

print("\nORIGINAL")
print("Summarization Reason:", summarization_metric.reason)
print("Clarity Reason:", clarity.reason)
print("Tonality Reason:", tonality.reason)
print("Safety Reason:", safety.reason)

print("\nIMPROVED")
print("Improved Summary:", improved_summary)
print("Summarization Score:", summarization_metric.score)
print("Clarity Score:", clarity.score)
print("Tonality Score:", tonality.score)
print("Safety Score:", safety.score)

Output()

Output()

Output()

Output()


ORIGINAL
Summarization Reason: The score is 0.00 because the summary contains significant contradictions to the original text, such as misrepresenting the return on investment for generative AI and incorrectly identifying the barriers to scaling AI. Additionally, the summary introduces extra information that is not present in the original text, which further detracts from its accuracy and relevance.
Clarity Reason: The summary is concise and logically structured, effectively outlining the key issues surrounding the 'GenAI Divide' without unnecessary repetition. It uses clear language, although some terms like 'Agentic Web' may require further explanation for clarity. Overall, it presents a coherent narrative that is easy to follow, but could benefit from a bit more clarity on specific jargon.
Tonality Reason: The summary is written in a professional and specific tone, using appropriate language throughout. It avoids slang and disrespectful expressions, maintaining a contextually appro

I got a better output overall (i.e., clarity and tonality improved), however not all issues were fixed. For example, there is still some hallucincation and contradictions happening in the summary after the enhancement. I noticed that the way I write the prompt matters a lot. It's important to be more specific and require strict rules. On top of that, we are using the same model to generate and evaluate/enhance the summary so I wouldn't expect drastic improvements. 

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
